# 🌿 AGRILENS — MobileNetV4 Retraining Pipeline
### Complete 27-Class Agricultural Kiosk Retraining on Google Colab (Free GPU)

This notebook trains a **MobileNetV4-Conv-Small** vision backbone on all 5 project crops + negative non-target foliage:
1. **Apple** (Rust, Scab, Healthy)
2. **Mango** (Anthracnose, Bacterial Canker, Cutting Weevil, Die Back, Gall Midge, Healthy, Powdery Mildew, Sooty Mould)
3. **Tomato** (Bacterial Spot, Early/Late Blight, Leaf Mold, Septoria, Spider Mites, Target Spot, TYLCV, Mosaic, Healthy)
4. **Pepper** (Bacterial Spot, Healthy)
5. **Potato** (Early Blight, Late Blight, Healthy)
6. **Other_Plant_or_Background** (Negative class for Money Plant, weeds, and non-target foliage)

**Hardware Requirement:** Free T4 GPU (Runtime -> Change runtime type -> T4 GPU).
**Training Time:** ~12–18 minutes for 15 epochs on T4 GPU.

In [ ]:
# 1. Install PyTorch Image Models (timm) and ONNX
!pip install -q timm onnx onnxruntime

In [ ]:
# 2. Verify Hardware Accelerator (GPU)
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU")

In [ ]:
# 3. Mount Google Drive to load your EXPISCOR 2026 dataset
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA_DIR = '/content/drive/MyDrive/EXPISCOR 2026/MobileV4/datasets'
print("Datasets present in Drive:", os.listdir(DRIVE_DATA_DIR))

In [ ]:
# 4. Fast Dataset Consolidation on High-Speed Local NVMe Disk (/content/dataset)
import zipfile, shutil
from pathlib import Path

LOCAL_DATA = Path('/content/dataset')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

# Unzip Apple
with zipfile.ZipFile(f'{DRIVE_DATA_DIR}/apple dataset.zip', 'r') as z:
    for m in z.namelist():
        for c, t in [('rust', 'Apple___Rust'), ('scab', 'Apple___Scab'), ('healthy', 'Apple___healthy')]:
            if f'/{c}/' in m.lower():
                d = LOCAL_DATA / t
                d.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(d / Path(m).name, 'wb') as dst:
                    dst.write(src.read())

# Unzip Mango
with zipfile.ZipFile(f'{DRIVE_DATA_DIR}/mango dataset.zip', 'r') as z:
    for m in z.namelist():
        top = m.split('/')[0].lower().strip()
        mapping = {
            'anthracnose': 'Mango___Anthracnose',
            'bacterial canker': 'Mango___Bacterial_Canker',
            'cutting weevil': 'Mango___Cutting_Weevil',
            'die back': 'Mango___Die_Back',
            'gall midge': 'Mango___Gall_Midge',
            'healthy': 'Mango___Healthy',
            'powdery mildew': 'Mango___Powdery_Mildew',
            'sooty mould': 'Mango___Sooty_Mould'
        }
        if top in mapping and m.endswith(('.jpg', '.jpeg', '.png', '.JPG')):
            d = LOCAL_DATA / mapping[top]
            d.mkdir(parents=True, exist_ok=True)
            with z.open(m) as src, open(d / Path(m).name, 'wb') as dst:
                dst.write(src.read())

# Copy Pepper & Potato from PlantVillage
pv_source = Path(f'{DRIVE_DATA_DIR}/PlantVillage')
if not pv_source.exists():
    pv_source = Path('/content/drive/MyDrive/EXPISCOR 2026/MobileV2/dataset/PlantVillage')

pv_map = {
    'Pepper__bell___Bacterial_spot': 'Pepper___Bacterial_spot',
    'Pepper__bell___healthy': 'Pepper___healthy',
    'Potato___Early_blight': 'Potato___Early_blight',
    'Potato___Late_blight': 'Potato___Late_blight',
    'Potato___healthy': 'Potato___healthy'
}
for src_name, dst_name in pv_map.items():
    s = pv_source / src_name
    if s.exists():
        d = LOCAL_DATA / dst_name
        d.mkdir(parents=True, exist_ok=True)
        for f in s.glob('*.*'):
            shutil.copy2(f, d / f.name)

# Setup Negative Class (Other_Plant_or_Background)
(LOCAL_DATA / 'Other_Plant_or_Background').mkdir(parents=True, exist_ok=True)

print("Consolidated classes count:", len(os.listdir(LOCAL_DATA)))

In [ ]:
# 5. Setup MobileNetV4 Model & DataLoaders with timm
import timm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(str(LOCAL_DATA))
classes = dataset.classes
num_classes = len(classes)
print(f"Total Images: {len(dataset)} | Classes ({num_classes}): {classes}")

# 80/20 Train/Val Split
val_len = int(0.2 * len(dataset))
train_ds, val_ds = random_split(dataset, [len(dataset) - val_len, val_len], generator=torch.Generator().manual_seed(42))
train_ds.dataset.transform = train_tf
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# Create MobileNetV4 with Pretrained Weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = timm.create_model('mobilenetv4_conv_small', pretrained=True, num_classes=num_classes)
model = model.to(device)
print("MobileNetV4 Model Loaded Successfully on", device)

In [ ]:
# 6. Train with Mixed Precision (AMP) & Cosine Annealing Scheduler
import torch.nn as nn
import time

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

EPOCHS = 15
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss, train_corr = 0.0, 0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(imgs)
            loss = criterion(preds, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)
        train_corr += (preds.argmax(1) == targets).sum().item()
    
    scheduler.step()
    train_acc = 100.0 * train_corr / len(train_ds)
    
    # Validate
    model.eval()
    val_loss, val_corr = 0.0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            with torch.amp.autocast('cuda'):
                preds = model(imgs)
                loss = criterion(preds, targets)
            val_loss += loss.item() * imgs.size(0)
            val_corr += (preds.argmax(1) == targets).sum().item()
    
    val_acc = 100.0 * val_corr / len(val_ds)
    print(f"Epoch [{epoch:02d}/{EPOCHS}] ({time.time()-t0:.1f}s) | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/best_mobilenetv4.pth')

print(f"\nTraining Complete! Peak Accuracy: {best_acc:.2f}%")

In [ ]:
# 7. Export Best Model to ONNX for Raspberry Pi / Windows Kiosk
model.load_state_dict(torch.load('/content/best_mobilenetv4.pth'))
model.eval()
model.to('cpu')

dummy = torch.randn(1, 3, 224, 224, dtype=torch.float32)
onnx_file = '/content/agrilens_retrained.onnx'
torch.onnx.export(
    model,
    dummy,
    onnx_file,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Save labels.txt
with open('/content/labels_27class.txt', 'w') as f:
    for c in classes:
        f.write(f"{c}\n")

# Copy directly into Google Drive
shutil.copy2(onnx_file, f'{DRIVE_DATA_DIR}/../agrilens.onnx')
shutil.copy2('/content/labels_27class.txt', f'{DRIVE_DATA_DIR}/../labels.txt')
print("Successfully exported and updated agrilens.onnx & labels.txt in your Google Drive!")